# Lab 10 - Learning the XOR Boolean Function Using an MLP

**Aim:** To implement Multilayer Perceptrons (MLPs) using Keras, PyTorch and low-level TensorFlow to learn the non-linear XOR Boolean function.


## XOR Dataset

| Input 1 | Input 2 | XOR Output |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y = np.array([[0], [1], [1], [0]], dtype=np.float32)

print('X =')
print(X)
print('y =')
print(y)


I0000 00:00:1786608230.616154   64155 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786608231.847391   64155 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


X =
[[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
y =
[[0.]
 [1.]
 [1.]
 [0.]]


The final settings used are 4 hidden neurons, Tanh activation, learning rate 0.05 and 2000 epochs. Binary cross-entropy is used as the loss function and Adam as the optimizer.


## 1. Keras (TensorFlow High-Level API)


In [2]:
def build_keras_model(hidden_units, learning_rate):
    model = keras.Sequential([
        keras.layers.Input(shape=(2,)),
        keras.layers.Dense(hidden_units, activation='tanh'),
        keras.layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

# Hyperparameter experiment: hidden neurons, learning rate and epochs.
experiments = [(2, 0.01, 1000), (4, 0.01, 1500), (4, 0.05, 2000)]
experiment_results = []

for hidden_units, learning_rate, epochs in experiments:
    keras.utils.set_random_seed(42)
    model = build_keras_model(hidden_units, learning_rate)
    model.fit(X, y, epochs=epochs, verbose=0)
    predictions = (model.predict(X, verbose=0) >= 0.5).astype(int)
    experiment_results.append([hidden_units, learning_rate, epochs, (predictions == y).mean()])

keras_experiments = pd.DataFrame(
    experiment_results,
    columns=['Hidden neurons', 'Learning rate', 'Epochs', 'Accuracy'],
)
print(keras_experiments.to_string(index=False))


 Hidden neurons  Learning rate  Epochs  Accuracy
              2           0.01    1000       0.5
              4           0.01    1500       1.0
              4           0.05    2000       1.0


In [3]:
hidden_units, learning_rate, epochs = 4, 0.05, 2000
keras.utils.set_random_seed(42)
keras_model = build_keras_model(hidden_units, learning_rate)
keras_history = keras_model.fit(X, y, epochs=epochs, verbose=0)

keras_probabilities = keras_model.predict(X, verbose=0)
keras_predictions = (keras_probabilities >= 0.5).astype(int)
keras_accuracy = (keras_predictions == y).mean()

keras_results = pd.DataFrame({
    'Input 1': X[:, 0].astype(int),
    'Input 2': X[:, 1].astype(int),
    'Expected': y.ravel().astype(int),
    'Predicted probability': keras_probabilities.ravel().round(4),
    'Predicted output': keras_predictions.ravel(),
})
print(keras_results.to_string(index=False))
print(f'Keras accuracy: {keras_accuracy:.2%}')
assert np.array_equal(keras_predictions, y.astype(int)), 'Keras did not learn XOR.'


 Input 1  Input 2  Expected  Predicted probability  Predicted output
       0        0         0                 0.0000                 0
       0        1         1                 0.9998                 1
       1        0         1                 0.9999                 1
       1        1         0                 0.0002                 0
Keras accuracy: 100.00%


## 2. PyTorch


In [4]:
torch.manual_seed(42)
X_torch = torch.tensor(X)
y_torch = torch.tensor(y)

class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 4),
            nn.Tanh(),
            nn.Linear(4, 1),
        )

    def forward(self, inputs):
        return self.network(inputs)

torch_model = XORNet()
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(torch_model.parameters(), lr=0.05)

for _ in range(2000):
    optimizer.zero_grad()
    loss = loss_function(torch_model(X_torch), y_torch)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    torch_probabilities = torch.sigmoid(torch_model(X_torch)).numpy()
torch_predictions = (torch_probabilities >= 0.5).astype(int)
torch_accuracy = (torch_predictions == y).mean()

torch_results = pd.DataFrame({
    'Input 1': X[:, 0].astype(int),
    'Input 2': X[:, 1].astype(int),
    'Expected': y.ravel().astype(int),
    'Predicted probability': torch_probabilities.ravel().round(4),
    'Predicted output': torch_predictions.ravel(),
})
print(torch_results.to_string(index=False))
print(f'PyTorch accuracy: {torch_accuracy:.2%}')
assert np.array_equal(torch_predictions, y.astype(int)), 'PyTorch did not learn XOR.'


 Input 1  Input 2  Expected  Predicted probability  Predicted output
       0        0         0                 0.0000                 0
       0        1         1                 0.9998                 1
       1        0         1                 0.9998                 1
       1        1         0                 0.0004                 0
PyTorch accuracy: 100.00%


## 3. TensorFlow Low-Level API


In [5]:
tf.random.set_seed(42)
X_tf = tf.constant(X)
y_tf = tf.constant(y)

W1 = tf.Variable(tf.random.normal([2, 4], stddev=0.5))
b1 = tf.Variable(tf.zeros([4]))
W2 = tf.Variable(tf.random.normal([4, 1], stddev=0.5))
b2 = tf.Variable(tf.zeros([1]))
optimizer = tf.optimizers.Adam(learning_rate=0.05)

def tf_logits(inputs):
    hidden = tf.nn.tanh(tf.matmul(inputs, W1) + b1)
    return tf.matmul(hidden, W2) + b2

for _ in range(2000):
    with tf.GradientTape() as tape:
        logits = tf_logits(X_tf)
        loss = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y_tf, logits=logits))
    gradients = tape.gradient(loss, [W1, b1, W2, b2])
    optimizer.apply_gradients(zip(gradients, [W1, b1, W2, b2]))

tf_probabilities = tf.sigmoid(tf_logits(X_tf)).numpy()
tf_predictions = (tf_probabilities >= 0.5).astype(int)
tf_accuracy = (tf_predictions == y).mean()

tf_results = pd.DataFrame({
    'Input 1': X[:, 0].astype(int),
    'Input 2': X[:, 1].astype(int),
    'Expected': y.ravel().astype(int),
    'Predicted probability': tf_probabilities.ravel().round(4),
    'Predicted output': tf_predictions.ravel(),
})
print(tf_results.to_string(index=False))
print(f'TensorFlow low-level accuracy: {tf_accuracy:.2%}')
assert np.array_equal(tf_predictions, y.astype(int)), 'Low-level TensorFlow did not learn XOR.'


 Input 1  Input 2  Expected  Predicted probability  Predicted output
       0        0         0                 0.0000                 0
       0        1         1                 0.9997                 1
       1        0         1                 0.9999                 1
       1        1         0                 0.0002                 0
TensorFlow low-level accuracy: 100.00%


## Comparison and Observation


In [6]:
comparison = pd.DataFrame({
    'Library': ['Keras', 'PyTorch', 'TensorFlow low-level'],
    'Final accuracy': [keras_accuracy, torch_accuracy, tf_accuracy],
})
print(comparison.to_string(index=False, formatters={'Final accuracy': '{:.2%}'.format}))


             Library Final accuracy
               Keras        100.00%
             PyTorch        100.00%
TensorFlow low-level        100.00%


All three MLPs learn the XOR function when the hidden layer introduces non-linearity. Too few epochs or a very small learning rate can prevent convergence, while an excessively large learning rate can make training unstable. More hidden neurons increase model capacity; for XOR, a small hidden layer is sufficient. Tanh (or ReLU) is needed because a network using only linear layers cannot learn the non-linear XOR pattern.
